In [4]:
# Import the necessary libraries
import os
from urllib.parse import quote_plus

import mysql.connector
import pandas as pd
from dotenv import load_dotenv
from mysql.connector import Error
from pymongo import MongoClient

load_dotenv()

True

In [5]:
# MySQL connection details from .env
hostname = os.getenv("hostnameMySQL")
database = os.getenv("databaseMySQL")
port = os.getenv("portMySQL", "3306")
username = os.getenv("usernameMySQL")
password = os.getenv("passwordMySQL")

required_values = {
    "hostnameMySQL": hostname,
    "databaseMySQL": database,
    "usernameMySQL": username,
    "passwordMySQL": password,
}
missing_values = [name for name, value in required_values.items() if not value]
if missing_values:
    raise ValueError(f"Missing required .env variables: {', '.join(missing_values)}")

connection = None
cursor = None
try:
    connection = mysql.connector.connect(
        host=hostname, database=database, user=username, password=password, port=int(port)
    )
    if connection.is_connected():
        print("Connected to MySQL Server version", connection.get_server_info())
        cursor = connection.cursor()
        cursor.execute("SELECT DATABASE();")
        print("Connected to database:", cursor.fetchone()[0])

except Error as error:
    print("Error while connecting to MySQL:", error)
finally:
    if cursor is not None:
        cursor.close()
    if connection is not None and connection.is_connected():
        connection.close()
        print("MySQL connection is closed")

/var/folders/hw/2p1y1pnn2ws2r2ylnxfmlkzm0000gn/T/ipykernel_89912/1210242779.py:25: DeprecationWarning: Call to deprecated function get_server_info. Reason: 
    The property counterpart 'server_info' should be used instead.

  print("Connected to MySQL Server version", connection.get_server_info())


Connected to MySQL Server version 8.0.36-28
Connected to database: olistproject_character
MySQL connection is closed


In [6]:

order_payments = pd.read_csv("data/olist_order_payments_dataset.csv")
order_payments.head()
#order_payments.shape

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [7]:
# Upload the payment dataset to MySQL
csv_file_path = "data/olist_order_payments_dataset.csv"
table_name = "olist_order_payments"

connection = None
cursor = None
try:
    connection = mysql.connector.connect(
        host=hostname,
        database=database,
        user=username,
        password=password,
        port=int(port),
    )
    cursor = connection.cursor()
    print("Connected to MySQL successfully!")

    cursor.execute(f"DROP TABLE IF EXISTS `{table_name}`;")
    cursor.execute(
        f"""
        CREATE TABLE `{table_name}` (
            order_id VARCHAR(50),
            payment_sequential INT,
            payment_type VARCHAR(20),
            payment_installments INT,
            payment_value DECIMAL(10, 2)
        );
        """
    )

    data = pd.read_csv(csv_file_path)
    batch_size = 500
    total_records = len(data)
    insert_query = f"""
        INSERT INTO `{table_name}`
        (order_id, payment_sequential, payment_type, payment_installments, payment_value)
        VALUES (%s, %s, %s, %s, %s);
    """

    for start in range(0, total_records, batch_size):
        end = min(start + batch_size, total_records)
        batch_records = list(data.iloc[start:end].itertuples(index=False, name=None))
        cursor.executemany(insert_query, batch_records)
        print(f"Prepared records {start + 1} to {end}")

    connection.commit()
    cursor.execute(f"SELECT COUNT(*) FROM `{table_name}`;")
    inserted_records = cursor.fetchone()[0]
    if inserted_records != total_records:
        raise RuntimeError(
            f"Row-count mismatch: CSV has {total_records}, MySQL has {inserted_records}"
        )
    print(f"All {inserted_records} records inserted into `{table_name}`.")

except Exception as error:
    if connection is not None:
        connection.rollback()
    print("Error while loading data into MySQL:", error)
    raise

finally:
    if cursor is not None:
        cursor.close()
    if connection is not None and connection.is_connected():
        connection.close()
        print("MySQL connection is closed.")

Connected to MySQL successfully!
Prepared records 1 to 500
Prepared records 501 to 1000
Prepared records 1001 to 1500
Prepared records 1501 to 2000
Prepared records 2001 to 2500
Prepared records 2501 to 3000
Prepared records 3001 to 3500
Prepared records 3501 to 4000
Prepared records 4001 to 4500
Prepared records 4501 to 5000
Prepared records 5001 to 5500
Prepared records 5501 to 6000
Prepared records 6001 to 6500
Prepared records 6501 to 7000
Prepared records 7001 to 7500
Prepared records 7501 to 8000
Prepared records 8001 to 8500
Prepared records 8501 to 9000
Prepared records 9001 to 9500
Prepared records 9501 to 10000
Prepared records 10001 to 10500
Prepared records 10501 to 11000
Prepared records 11001 to 11500
Prepared records 11501 to 12000
Prepared records 12001 to 12500
Prepared records 12501 to 13000
Prepared records 13001 to 13500
Prepared records 13501 to 14000
Prepared records 14001 to 14500
Prepared records 14501 to 15000
Prepared records 15001 to 15500
Prepared records 15

In [ ]:
# MongoDB connection details from .env
mongo_hostname = os.getenv("hostnameMongoDB")
mongo_database = os.getenv("databaseMongoDB")
mongo_port = os.getenv("portMongoDB")
mongo_username = os.getenv("usernameMongoDB")
mongo_password = os.getenv("passwordMongoDB")

mongo_uri = (
    f"mongodb://{quote_plus(mongo_username)}:{quote_plus(mongo_password)}"
    f"@{mongo_hostname}:{mongo_port}/{mongo_database}"
)


In [ ]:
# Upload product-category translations to MongoDB
product_category_df = pd.read_csv(
    "data/product_category_name_translation.csv", encoding="utf-8-sig"
)
client = None
try:
    client = MongoClient(mongo_uri)
    collection = client[mongo_database]["product_categories"]
    records = product_category_df.to_dict(orient="records")
    result = collection.insert_many(records)
    print(f"Uploaded {len(result.inserted_ids)} records to MongoDB successfully!")
except Exception as error:
    print(f"Error while loading data into MongoDB: {error}")
    raise

finally:
    if client is not None:
        client.close()
        print("MongoDB connection is closed")